# Support Vector Machines

**Support Vector Machines (SVMs)** are a powerful tool for **classification** and **regression** tasks. In their basic formulation, SVMs are **binary classifiers** that learn a **linear decision boundary** in the form of a hyperplane that separates two classes of data. While SVMs can be extended to handle non-linear decision boundaries, multiclass classification, and regression, these capabilities are not part of the core formulation and rely on additional techniques.

SVMs construct this decision boundary by learning a **linear separator** in the **feature space**. In two dimensions, this separator is a **line**, while in higher dimensions it is a **hyperplane**. Among all possible hyperplanes that could separate the data, SVMs select the one that achieves the **maximum margin**.

The **margin** is the distance between the hyperplane and the nearest data points from each class. These closest points are known as **support vectors**, and they are the critical elements that determine the position and orientation of the hyperplane. By maximizing this margin, the SVM chooses a decision boundary that lies as far as possible from both classes, improving robustness to small variations or noise in the data and reducing the risk of overfitting.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import plotly

import sklearn.base
import sklearn.pipeline
import sklearn.svm
import sklearn.utils.validation

from IPython.display import display, HTML

import sys
import os
sys.path.append(os.path.abspath("src"))
import tutorial

## Linearly Separable Data

To build intuition, we begin with a dataset that is _linearly separable_, meaning there exists at least one hyperplane that can perfectly separate the two classes.

In [ ]:
linearly_data_df = tutorial.generate_separable_dataset()
tutorial.plot_data_2d(linearly_data_df, features=["x", "y"], target="target")

From the above plot, it is clear that there exists a straight line can partition the two classes of data points, and so the data is linearly separable. In this setting, an SVM is capable of identifying a decision boundary that not only separates the classes but also maximizes the margin between them.

In general, the learned decision boundary can be expressed as a hyperplane of the form $w^T \vec{x} + b = 0$, where $w$ represents the weight vector and $b$ is the intercept. These parameters define the orientation and position of the hyperplane in the feature space.

To make a prediction for a new data point $\vec{x}$, we evaluate the expression $w^T \vec{x} + b$. The sign of this value determines on which side of the hyperplane the point lies. If $w^T \vec{x} + b > 0$, the point is assigned to one class, and if $w^T \vec{x} + b < 0$, it is assigned to the other class. Points that lie exactly on the hyperplane satisfy $w^T \vec{x} + b = 0$ and represent the decision boundary itself.

We can use the [SVM (sklearn.svm.SVC)](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) implementation from scikit-learn to find this maximum-margin decision boundary along with the corresponding support vectors. The weight vector $w$ is given by the trained SVC’s `coef_` attribute, and the intercept $b$ is given by `intercept_`.

In [ ]:
svm = sklearn.svm.SVC(kernel="linear", C=1e6)
svm.fit(
    linearly_data_df[["x", "y"]], linearly_data_df["target"]
)

print("Support Vectors:")
print(*svm.support_vectors_, sep="\n")
print()

w = svm.coef_[0]
b = svm.intercept_[0]
print("Decision Boundary:")
print(f"{w[0]:0.3f}x + {w[1]:0.3f}y + {b:0.3f} = 0")

tutorial.plot_data_2d(linearly_data_df, features=["x", "y"], target="target", classifier=svm)

## Non-Linearly Separable Data

In the plot above, the maximum-margin hyperplane that separates the two classes is shown, along with the support vectors, which are highlighted by black circles. We see that the hyperplane perfectly separates the data.

This raises an important question: what happens when the data is not linearly separable, and no hyperplane can perfectly separate the classes? To explore this case, consider the following dataset:

In [ ]:
nonlinearly_data_df = tutorial.generate_non_separable_dataset()
tutorial.plot_data_2d(nonlinearly_data_df, features=["x", "y"], target="target")

Although there appears to be some structure or relationship in the data, it is not linearly separable. If we try to use an SVM to classify this data directly in the original feature space, we would not expect it to perform well.

The result of fitting an SVM to this dataset illustrates how a linear decision boundary behaves when the data is not linearly separable. As we can see, the SVM struggles to separate the classes cleanly, highlighting the limitations of a linear hyperplane in this instance.

In [ ]:
svm = sklearn.svm.SVC(kernel="linear")
svm.fit(
    nonlinearly_data_df[["x", "y"]], nonlinearly_data_df["target"]
)

tutorial.plot_data_2d(
    nonlinearly_data_df, features=["x", "y"], target="target", classifier=svm, highlight_support=False
)

Despite the fact that the data is not linearly separable, there is still a clear structure that distinguishes the two classes. Specifically, one class of points appears to be enclosed within a circle, separating it from the other class of points.

Upon closer inspection, the inner class of points is centered at $(0,0)$. This suggests a way to transform the data into a higher-dimensional space where it may become linearly separable.

Consider the following mapping:

$$(x, y) \mapsto \left(x, y, \sqrt{x^2 + y^2}\right)$$


The intuition behind this mapping is that each point is lifted in the $z$-direction according to its distance from the origin. Points closer to $(0,0)$ remain near the $xy$-plain, while points farther away are mapped higher. Applying this transformation to all of the points produces the three-dimensional representation shown below:

In [ ]:
class RadialDistanceTransformer(sklearn.base.BaseEstimator, sklearn.base.TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = sklearn.utils.validation.check_array(X)

        x = X[:, 0]
        y = X[:, 1]

        z = np.sqrt(x**2 + y**2)

        return np.column_stack((x, y, z))

transformer = RadialDistanceTransformer()

X = transformer.fit_transform(
    nonlinearly_data_df[["x", "y"]].values
)

transformed_df = pd.DataFrame(
    X, columns=["x", "y", "z"]
)

transformed_df["target"] = nonlinearly_data_df["target"].values

tutorial.plot_data_3d(
    transformed_df,
    features=["x", "y", "z"],
    target="target"
)

In this transformed space, the points are mapped onto a cone, with one class concentrated near the tip and the other distributed farther out. The data is now linearly separable, meaning there exists a plane that can perfectly separate the two classes. An SVM can be applied in this space to identify the maximum-margin separating hyperplane as follows:

In [ ]:
svm = sklearn.svm.SVC(kernel="linear", C=1e6)
svm.fit(
    transformed_df[["x", "y", "z"]], transformed_df["target"]
)

print("Support Vectors:")
print(*svm.support_vectors_, sep="\n")
print()

w = svm.coef_[0]
b = svm.intercept_[0]
print("Decision Boundary:")
print(f"{w[0]:0.3f}x + {w[1]:0.3f}y + {w[2]:0.3f}z + {b:0.3f} = 0")

tutorial.plot_data_3d(
    transformed_df, features=["x", "y", "z"], target="target", classifier=svm
)

In scikit-learn, this process can be streamlined by avoiding the need to manually apply the transformation before fitting the model. Using a [sklearn.pipeline.Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), the transformation and the SVM can be combined into a single pipeline, ensuring that the same mapping is applied consistently during both training and prediction. This approach simplifies the implementation, reduces the risk of errors, and keeps preprocessing and model fitting organized within a single reusable object.

In [ ]:
pipeline = sklearn.pipeline.Pipeline([
    ("radial_feature", RadialDistanceTransformer()),
    ("svm", sklearn.svm.SVC(kernel="linear", C=1e6))
])

pipeline.fit(
    nonlinearly_data_df[["x", "y"]], nonlinearly_data_df["target"]
)

# attach original-space samples corresponding to the SVM support vectors
# so they can be plotted in the original feature space.
pipeline.support_vectors_ = nonlinearly_data_df[["x", "y"]].values[
    pipeline.named_steps["svm"].support_
]

tutorial.plot_data_2d(
    nonlinearly_data_df, features=["x", "y"], target="target", classifier=pipeline, highlight_support=True
)

In the above plot, we can see the decision boundary that the SVM learned in the higher-dimensional space, projected back into the original feature space. Notice that this boundary takes the shape of a circle, as expected. This boundary is non-linear because the transformation we applied to the data is non-linear.

More generally, when mapping data into a new feature space before applying an SVM, a linear decision boundary in the transformed space corresponds to a non-linear decision boundary in the original space if the transformation itself is non-linear. Conversely, if the transformation is linear, the resulting decision boundary in the original space will also be linear.

It is also worth noting that applying a linear transformation to the data does not improve classification performance, since it does not make the data more separable. However, such transformations can still be useful in practice for reasons such as reducing computational cost or memory usage.

## The Kernel Trick

In the previous example, when we projected the data into a higher-dimensional space, we incurred additional memory and computational overhead. Ideally, we would like to avoid explicitly performing this transformation, and it turns out that this is possible.

An SVM does not require explicit access to the feature vectors in order to learn a separating hyperplane. Instead, due to the way the optimization problem is formulated, it depends only on **dot products** between data points in the feature space. This means that if we can compute the dot product in the transformed feature space directly, without explicitly computing the transformation itself, then we have all the information needed to train and evaluate the SVM. This observation is the key idea behind the **kernel trick**.

Recall that the dot product between two vectors in $\mathbb{R}^n$ is defined as the sum of the products of their corresponding components. For vectors $\vec{u} = (u_{1}, u_{2}, \dots, u_{n})$ and $\vec{v} = (v_{1}, v_{2}, \dots, v_{n})$, the dot product is given by:

$$
\vec{u} \cdot \vec{v} = \sum_{i=1}^{n} u_{i} v_{i}
$$

Let $\phi$ denote our previously defined transformation:

$$
(x, y) \mapsto \left(x, y, \sqrt{x^{2} + y^{2}}\right)\,.
$$

If we have two vectors, say $\vec{v}_{1}=(x_{1},y_{1})$ and $\vec{v}_{2}=(x_{2},y_{2})$, then their dot product in the original space is:

$$
\begin{aligned}
\vec{v}_{1} \cdot \vec{v}_{2}
&= (x_{1},y_{1}) \cdot (x_{2},y_{2}) \\
&= x_{1} x_{2} + y_{1} y_{2}\,.
\end{aligned}
$$

However, under the transformation $\phi$, it becomes:

$$
\begin{aligned}
\phi(\vec{v}_{1}) \cdot \phi(\vec{v}_{2})
&= \left(x_{1}, y_{1}, \sqrt{x_{1}^{2} + y_{1}^{2}}\right) \cdot \left(x_{2}, y_{2}, \sqrt{x_{2}^{2} + y_{2}^{2}}\right) \\
&= x_{1} x_{2} + y_{1} y_{2} + \sqrt{x_{1}^{2} + y_{1}^{2}}\,\sqrt{x_{2}^{2} + y_{2}^{2}} \\
&= x_{1} x_{2} + y_{1} y_{2} + \sqrt{(x_{1}^{2} + y_{1}^{2})(x_{2}^{2} + y_{2}^{2})}\,.
\end{aligned}
$$

Notice that this expression depends only on the original coordinates and does not require explicitly computing $\phi(v_1)$ or $\phi(v_2)$. A function that computes such quantities is an example of a **kernel function** (_more on these later_). Using kernel functions, SVMs can behave as if the data were mapped into a higher-dimensional space without ever performing the transformation explicitly. This computational shortcut, which saves both time and memory, is known as the **kernel trick**.

**Note:** These ideas are part of a broader class of techniques known as **kernel methods**, which appear in many algorithms beyond SVMs.

Let’s now apply the kernel trick to our previous example. Instead of explicitly mapping the data into a higher-dimensional space, we can define a kernel function that computes the dot product in the transformed space directly. This kernel can then be passed to [sklearn.svm.SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) via its `kernel` parameter, allowing the SVM to operate without ever explicitly creating new features.

In [ ]:
def RadialFeatureKernel(U, V):
    U = sklearn.utils.validation.check_array(U)
    V = sklearn.utils.validation.check_array(V)

    x1 = U[:, 0][:, None]
    y1 = U[:, 1][:, None]

    x2 = V[:, 0][None, :]
    y2 = V[:, 1][None, :]
    
    return x1*x2 + y1*y2 + np.sqrt((x1**2 + y1**2) * (x2**2 + y2**2))

svm = sklearn.svm.SVC(kernel=RadialFeatureKernel, C=1e6)
svm.fit(
    nonlinearly_data_df[["x", "y"]], nonlinearly_data_df["target"]
)

# attach original-space samples corresponding to the SVM support vectors
# so they can be plotted in the original feature space.
svm.support_vectors_ = nonlinearly_data_df[["x", "y"]].values[svm.support_]

tutorial.plot_data_2d(
    nonlinearly_data_df, features=["x", "y"], target="target", classifier=svm, highlight_support=True
)

It is worth emphasizing that the mapping we defined was chosen by inspecting the data and reasoning about a transformation that would make the classes linearly separable. In particular, this mapping worked because the structure of the data (specifically its center) aligned with the transformation. If the data had been shifted, this same mapping would no longer be appropriate. Although we could still design a different transformation that works, in practice this becomes increasingly difficult, especially when the data is high-dimensional and not easily visualized.


Let us now examine what happens if we apply our original mapping to a shifted version of the data.

In [ ]:
nonlinearly_data_df2 = nonlinearly_data_df.copy()
nonlinearly_data_df2[["x", "y"]] = nonlinearly_data_df2[["x", "y"]] + 1

tutorial.plot_data_2d(nonlinearly_data_df2, features=["x", "y"], target="target")

X = transformer.fit_transform(nonlinearly_data_df2[["x", "y"]])

transformed_df2 = pd.DataFrame(X, columns=["x", "y", "z"])
transformed_df2["target"] = nonlinearly_data_df2["target"].values

tutorial.plot_data_3d(
    transformed_df2,
    features=["x", "y", "z"],
    target="target"
)

We can see that, while the original points appear unchanged except for a shift along the $xy$-axis, their mapping into the new space (onto the cone) differs significantly. After applying the same transformation to the shifted data, the points still lie on a cone; however, the relative arrangement of the two classes changes noticeably. In particular, the positive class is no longer concentrated near the tip of the cone. As a result, the data is no longer linearly separable in the transformed space, as can be seen below:

In [ ]:
svm = sklearn.svm.SVC(kernel="linear")
svm.fit(
    transformed_df2[["x", "y", "z"]], transformed_df["target"]
)

tutorial.plot_data_3d(
    transformed_df2, features=["x", "y", "z"], target="target", classifier=svm, highlight_support=False
)

We can see from the above plot that the SVM does a reasonably good job of separating the points; however, it does not achieve perfect separation.

This is possible because the scikit-learn implementation of the SVM does not require the data to be perfectly separable. Instead, it uses an extension that allows for some misclassifications through the use of a **soft margin**; this introduces a trade-off between maximizing the margin and minimizing classification errors. This flexibility enables the model to learn a useful separating hyperplane even when the data is not perfectly linearly separable.

Let us now examine how the learned decision boundary looks in the original space:

In [ ]:
svm = sklearn.svm.SVC(kernel=RadialFeatureKernel, C=1e6)
svm.fit(
    nonlinearly_data_df2[["x", "y"]], nonlinearly_data_df2["target"]
)

# attach original-space samples corresponding to the SVM support vectors
# so they can be plotted in the original feature space.
svm.support_vectors_ = nonlinearly_data_df2[["x", "y"]].values[svm.support_]

tutorial.plot_data_2d(
    nonlinearly_data_df2, features=["x", "y"], target="target", classifier=svm, highlight_support=False
)

## The Power of Kernels

In many real-world scenarios, it is not feasible to inspect the data and manually determine a suitable non-linear feature mapping. This is where **kernel methods** become useful: they allow SVMs to operate as if the data were mapped into a higher-dimensional space without explicitly performing the transformation.

A **kernel** is a function that behaves like a dot product in some (possibly high-dimensional) space. More formally, any function $K(u,v)$ that is **continuous**, **symmetric**, and **positive semidefinite** corresponds to a dot product in some feature space and can therefore be used as a kernel in an SVM.

The exact meaning of these requirements is not important for practical use, and the details are postponed to section [Kernel Theory Details (Supplementary)](#Kernel-Theory-Details-(Supplementary)). In practice, one rarely needs to derive new kernels from scratch. Instead, one relies on a standard set of kernels that are already known to satisfy these properties, such as:

- **Linear Kernel:**  
  $$K(\vec{u}, \vec{v}) = \vec{u} \cdot \vec{v}$$  
  Produces a linear decision boundary, as in the standard SVM.

- **Polynomial Kernel (degree $d$):**  
  $$K(\vec{u}, \vec{v}) = (\vec{u} \cdot \vec{v} + c)^{d}$$  
  Introduces curved decision boundaries by incorporating interactions between features.

- **Radial Basis Function (RBF) / Gaussian Kernel:**  
  $$K(\vec{u}, \vec{v}) = \exp\Big(-\gamma \|\vec{u} - \vec{v}\|^{2}\Big)$$  
  Captures complex, non-linear relationships by grouping nearby points together.

- **Sigmoid Kernel:**  
  $$K(\vec{u}, \vec{v}) = \tanh(\alpha \, \vec{u} \cdot \vec{v} + c)$$  
  Inspired by neural networks, though less commonly used in practice.

These kernels are widely used because they satisfy the necessary properties and allow SVMs to model non-linear decision boundaries without manual feature engineering.

For our non-linearly separable dataset, we will use the **RBF kernel**. This kernel is particularly well-suited for situations in which one class of points is enclosed by another, as it can produce **circular or radial decision boundaries** in the original feature space.

In [ ]:
svm = sklearn.svm.SVC(kernel="rbf", C=1e6)
svm.fit(
    nonlinearly_data_df2[["x", "y"]], nonlinearly_data_df2["target"]
)

# attach original-space samples corresponding to the SVM support vectors
# so they can be plotted in the original feature space.
svm.support_vectors_ = nonlinearly_data_df2[["x", "y"]].values[svm.support_]

tutorial.plot_data_2d(
    nonlinearly_data_df2, features=["x", "y"], target="target", classifier=svm, highlight_support=True
)

Our original non-linear dataset could be separated because we carefully inspected it and designed a mapping that made it linearly separable. However, when the data was shifted, that same mapping no longer worked. While it would have been possible to devise a new mapping by hand, we instead relied on the RBF kernel, which automatically provides a suitable non-linear transformation. Using the RBF kernel, the SVM was able to separate the classes effectively without the need to explicitly construct a new mapping, demonstrating the practical power and flexibility of kernel methods.

## Kernel Theory Details (Supplementary)

It is often impractical to explicitly define mappings from one vector space to another in order to make the data linearly separable. Ideally, we would like to specify a function that "just works," behaving like a dot product in some transformed space without needing to define or even understand the mapping itself. This section formalizes the requirements that allow such functions to behave like dot products in some vector space.

### Preliminaries

Before defining kernels, we provide the necessary background in vector spaces and inner products.

**Definition 1.**

For vectors $\vec{u}$ and $\vec{v}$ in $\mathbb{R}^{n}$, the **dot product** is defined as  

$$
\vec{u} \cdot \vec{v} = \sum_{i=1}^{n} u_{i} v_{i}
$$  

It is linear in each argument, symmetric, and positive definite.

**Example 1.**

Let $\vec{u} = (1, 2)$ and $\vec{v} = (3, 4)$ be vectors in $\mathbb{R}^{2}$. Then $\vec{u} \cdot \vec{v} = 1*3 + 2*4 = 11$.

**Definition 2.**

A real-valued **inner product** is a function $\langle \cdot, \cdot \rangle : V \times V \to \mathbb{R}$ on a vector space $V$ that satisfies the following properties for all $\vec{u}, \vec{v}, \vec{w} \in V$ and scalars $\alpha \in \mathbb{R}$:  

1. **Linearity:** $\langle \alpha \vec{u} + \vec{v}, \vec{w} \rangle = \alpha \langle \vec{u}, \vec{w} \rangle + \langle \vec{v}, \vec{w} \rangle$  
2. **Symmetry:** $\langle \vec{u}, \vec{v} \rangle = \langle \vec{v}, \vec{u} \rangle$  
3. **Positive definiteness:** $\langle \vec{v}, \vec{v} \rangle \ge 0$, with equality if and only if $\vec{v} = \vec{0}$  

The inner product generalizes the dot product to arbitrary vector spaces, including spaces that are not $\mathbb{R}^{n}$. Positive definiteness ensures that the "length" of any vector is non-negative.

**Example 2.**  

Let $V = \mathbb{R}^{2}$ and define the binary function $\langle \cdot, \cdot \rangle : V \times V \to \mathbb{R}$ by

$$
\langle \vec{u}, \vec{v} \rangle = 2 u_{1} v_{1} + 3 u_{2} v_{2}
$$

for all vectors $\vec{u} = (u_{1}, u_2)$ and $\vec{v} = (v_{1}, v_2)$ in $V$.

Let $\vec{u} = (1,2)$ and $\vec{v} = (3,4)$. Then

$$
\langle \vec{u}, \vec{v} \rangle = 2 \times 1 \times 3 + 3 \times 2 \times 4 = 6 + 24 = 30
$$

Next, consider arbitrary vectors $\vec{u}, \vec{v}, \vec{w} \in V$ and an arbitrary scalar $\alpha \in \mathbb{R}$. Then:

- **Linearity:**  
  $$
  \langle \alpha \vec{u} + \vec{w}, \vec{v} \rangle = 2 (\alpha u_{1} + w_{1})v_{1} + 3 (\alpha u_{2}+ w_{2})v_{2}= \alpha \langle \vec{u}, \vec{v} \rangle + \langle \vec{w}, \vec{v} \rangle
  $$
  so linearity is established.

- **Symmetry:**  
  $$
  \langle \vec{u}, \vec{v} \rangle = 2 u_{1} v_{1} + 3 u_{2}v_{2}= 2 v_{1} u_{1} + 3 v_{2}u_{2}= \langle \vec{v}, \vec{u} \rangle
  $$
  so symmetry is established.

- **Positive definiteness:**  
  $$
  \langle \vec{v}, \vec{v} \rangle = 2 v_{1}^2 + 3 v_2^2 \ge 0
  $$
  with equality only if $v_{1} = v_{2}= 0$.

Thus, this function defines a valid inner product on $V$. Conceptually, this inner product is a dot product that **weights the coordinates differently from the standard dot product**.

**Theorem (Equivalence of Inner Products and Dot Products in Finite Dimensions).**

Let $V$ be a finite-dimensional vector space with an inner product $\langle \cdot, \cdot \rangle$. Then there exists a linear transformation $T: V \to \mathbb{R}^n$ such that for all $\vec{u}, \vec{v} \in V$  

$$
\langle \vec{u}, \vec{v} \rangle = (T\vec{u}) \cdot (T\vec{v})
$$

This shows that any inner product in a finite-dimensional space is equivalent to a standard dot product after a suitable linear transformation. In infinite-dimensional spaces, inner products can similarly be interpreted as dot products in a Hilbert space.

### Mercer’s Theorem

Let $K(\vec{u}, \vec{v})$ be a function on a vector space satisfying the following three properties:

1. **Continuity**  
2. **Symmetry:** $K(\vec{u}, \vec{v}) = K(\vec{v}, \vec{u})$  
3. **Positive semidefinite:** For any finite set of vectors $\{\vec{x}_{1}, \dots, \vec{x}_{n}\}$, the matrix with entries $K_{ij} = K(\vec{x}_{i}, \vec{x}_{j})$ is positive semidefinite  

Then there exists a (possibly infinite-dimensional) vector space and a mapping $\phi$ such that  

$$
K(\vec{u}, \vec{v}) = \langle \phi(\vec{u}), \phi(\vec{v}) \rangle\,.
$$

Mercer’s Theorem is a key result that lets us define kernel functions that behave like dot products in some feature space, without ever needing to explicitly construct or understand the mapping; it _guarantees that such a mapping exists_, which is all that is need for SVMs and other kernel methods.

### Kernel Definition

A **kernel** is a real-valued, binary function defined on a vector space that is continuous, symmetric, and positive semidefinite.

Mercer’s Theorem guarantees that every kernel corresponds to an inner product in some (possibly infinite-dimensional) vector space. Furthermore, by the equivalence result from the preliminaries, there exists a linear transformation that converts this inner product into a standard dot product. This ensures that any kernel can be treated as a dot product in a transformed space, which is exactly what SVMs require.

> It is worth mentioning that all of these ideas can also be extended to complex vector spaces, allowing for **complex-valued kernels**.